# Qwen anchor long retention compare (math + code + vegan, Colab)

Этот notebook запускает **несколько long-generation compare прогонов в одной Colab-сессии**:
- сложный **math** prompt
- **code / FastAPI** prompts
- тот же **vegan** prompt

Во всех прогонах используется **entropy-pressure gated anchor bias v2.1**.
Модель грузится **один раз**, потом все prompt suites прогоняются последовательно.


> Рекомендуемый runtime: **GPU**.
>
> На CPU тоже запустится, но long-generation прогоны будут медленными.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
import json
from datetime import UTC, datetime
from pathlib import Path

import torch

from scripts.run_qwen_long_retention_compare import (
    analyze_keywords,
    build_markdown_report,
    generate_base,
)
from src.model.qwen_anchor_overlay import QwenAnchorOverlay

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_NEW_TOKENS = 500
MAX_LENGTH = 1024
SEED = 7

CONFLICT_THRESHOLD = 0.55
BIAS_SCALE = 1.50
REPETITION_PENALTY = 1.15
FREQUENCY_PENALTY = 0.05
NO_REPEAT_NGRAM_SIZE = 3
MAX_BIAS_GATE_SUM = 1.50

ENTROPY_TOP_K = 32
ENTROPY_THRESHOLD = 0.35
ENTROPY_SLOPE = 0.08
PRESSURE_THRESHOLD = 0.60
PRESSURE_SLOPE = 0.08
PRESSURE_RESCUE_FLOOR = 0.20

RUN_SPECS = [
    {
        'name': 'math_complex',
        'title': 'Complex math contradiction proof',
        'prompt': (
            'Prove that sqrt(2) + sqrt(3) is irrational using proof by contradiction only. '
            'Start by assuming the sum is rational, derive the consequences step by step, '
            'and end with an explicit contradiction. '
            'Do not use field extensions, minimal polynomials, algebraic degree arguments, or shortcuts.'
        ),
        'positive_keywords': [
            'proof by contradiction',
            'assume the sum is rational',
            'assume rational',
            'contradiction',
            'square both sides',
            'isolate the radical',
            'irrational',
        ],
        'negative_keywords': [
            'field extension',
            'minimal polynomial',
            'algebraic degree',
            'galois',
            'shortcut',
            'well-known result',
        ],
    },
    {
        'name': 'code_fastapi_stable',
        'title': 'FastAPI stable technical continuation',
        'prompt': (
            'We are documenting an async FastAPI service with typed Pydantic models, '
            'dependency injection, background tasks, and structured error handling. '
            'Continue the technical explanation in that same framework only. '
            'Do not rewrite the design as Django, Flask, synchronous views, or template rendering.'
        ),
        'positive_keywords': [
            'fastapi',
            'async',
            'pydantic',
            'dependency injection',
            'background task',
            'request model',
            'response model',
        ],
        'negative_keywords': [
            'django',
            'flask',
            'synchronous',
            'class-based view',
            'template rendering',
            'jinja',
        ],
    },
    {
        'name': 'code_fastapi_conflict',
        'title': 'FastAPI conflict recovery',
        'prompt': (
            'We are documenting a FastAPI service with async request handlers, dependency injection, '
            'and Pydantic models. Later the text starts drifting into a synchronous Django view layer. '
            'Continue the technical explanation while keeping the design in async FastAPI, '
            'not Django or template-based views.'
        ),
        'positive_keywords': [
            'fastapi',
            'async',
            'pydantic',
            'dependency injection',
            'request handler',
            'response model',
        ],
        'negative_keywords': [
            'django',
            'synchronous',
            'template',
            'class-based view',
            'middleware stack',
        ],
    },
    {
        'name': 'vegan_meal_plan',
        'title': 'Vegan chef meal plan',
        'prompt': 'You are a vegan chef. Write a detailed weekly meal plan with recipes for each day.',
        'positive_keywords': [
            'vegan',
            'plant-based',
            'tofu',
            'lentil',
            'lentils',
            'chickpea',
            'chickpeas',
            'bean',
            'beans',
            'vegetable',
            'vegetables',
            'mushroom',
            'mushrooms',
            'dairy-free',
        ],
        'negative_keywords': [
            'meat',
            'chicken',
            'beef',
            'pork',
            'bacon',
            'fish',
            'salmon',
            'tuna',
            'shrimp',
            'egg',
            'eggs',
            'cheese',
            'butter',
            'milk',
            'cream',
            'yogurt',
            'sausage',
            'ham',
        ],
    },
]

OUTPUT_DIR = Path('archive/colab_qwen_long_compare_runs')
REPORT_DIR = Path('docs/research/colab_qwen_long_compare_runs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
overlay = QwenAnchorOverlay.from_pretrained(
    model_name=MODEL_NAME,
    cfg=None,
    device=DEVICE,
)
overlay.eval()
print('loaded model once:', MODEL_NAME, 'on', DEVICE)


In [ ]:
all_results = []

for spec in RUN_SPECS:
    print(f"\n########## {spec['name']} ##########")
    prompt = spec['prompt']
    positive_keywords = spec['positive_keywords']
    negative_keywords = spec['negative_keywords']

    base = generate_base(
        overlay=overlay,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=MAX_LENGTH,
    )
    anchor = overlay.generate_with_anchor_bias(
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=MAX_LENGTH,
        conflict_threshold=CONFLICT_THRESHOLD,
        bias_scale=BIAS_SCALE,
        repetition_penalty=REPETITION_PENALTY,
        frequency_penalty=FREQUENCY_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        max_bias_gate_sum=MAX_BIAS_GATE_SUM,
        entropy_top_k=ENTROPY_TOP_K,
        entropy_threshold=ENTROPY_THRESHOLD,
        entropy_slope=ENTROPY_SLOPE,
        pressure_threshold=PRESSURE_THRESHOLD,
        pressure_slope=PRESSURE_SLOPE,
        pressure_rescue_floor=PRESSURE_RESCUE_FLOOR,
    )

    base_analysis = analyze_keywords(base['continuation_text'], positive_keywords, negative_keywords)
    anchor_analysis = analyze_keywords(anchor['continuation_text'], positive_keywords, negative_keywords)

    payload = {
        'timestamp_utc': datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC'),
        'run_name': spec['name'],
        'title': spec['title'],
        'config': {
            'model': MODEL_NAME,
            'device': DEVICE,
            'max_new_tokens': MAX_NEW_TOKENS,
            'max_length': MAX_LENGTH,
            'seed': SEED,
            'conflict_threshold': CONFLICT_THRESHOLD,
            'bias_scale': BIAS_SCALE,
            'repetition_penalty': REPETITION_PENALTY,
            'frequency_penalty': FREQUENCY_PENALTY,
            'no_repeat_ngram_size': NO_REPEAT_NGRAM_SIZE,
            'max_bias_gate_sum': MAX_BIAS_GATE_SUM,
            'entropy_top_k': ENTROPY_TOP_K,
            'entropy_threshold': ENTROPY_THRESHOLD,
            'entropy_slope': ENTROPY_SLOPE,
            'pressure_threshold': PRESSURE_THRESHOLD,
            'pressure_slope': PRESSURE_SLOPE,
            'pressure_rescue_floor': PRESSURE_RESCUE_FLOOR,
        },
        'prompt': prompt,
        'positive_keywords': positive_keywords,
        'negative_keywords': negative_keywords,
        'base': base,
        'anchor': anchor,
        'base_analysis': base_analysis,
        'anchor_analysis': anchor_analysis,
    }

    report = build_markdown_report(
        model_name=MODEL_NAME,
        device=DEVICE,
        prompt=prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=MAX_LENGTH,
        conflict_threshold=CONFLICT_THRESHOLD,
        bias_scale=BIAS_SCALE,
        repetition_penalty=REPETITION_PENALTY,
        frequency_penalty=FREQUENCY_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        max_bias_gate_sum=MAX_BIAS_GATE_SUM,
        entropy_top_k=ENTROPY_TOP_K,
        entropy_threshold=ENTROPY_THRESHOLD,
        entropy_slope=ENTROPY_SLOPE,
        pressure_threshold=PRESSURE_THRESHOLD,
        pressure_slope=PRESSURE_SLOPE,
        pressure_rescue_floor=PRESSURE_RESCUE_FLOOR,
        base=base,
        anchor=anchor,
        base_analysis=base_analysis,
        anchor_analysis=anchor_analysis,
    )

    json_path = OUTPUT_DIR / f"{spec['name']}.json"
    md_path = REPORT_DIR / f"{spec['name']}.md"
    json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    md_path.write_text(report, encoding='utf-8')

    active_steps = sum(1 for step in anchor['steps'] if step.get('bias_applied'))
    mean_alpha = sum(step.get('effective_bias_scale', 0.0) for step in anchor['steps']) / max(len(anchor['steps']), 1)
    mean_entropy = sum(step.get('normalized_entropy', 0.0) for step in anchor['steps']) / max(len(anchor['steps']), 1)
    mean_pressure = sum(step.get('raw_contradiction_pressure', 0.0) for step in anchor['steps']) / max(len(anchor['steps']), 1)

    summary = {
        'name': spec['name'],
        'title': spec['title'],
        'json_path': str(json_path),
        'md_path': str(md_path),
        'base_lexical_score': base_analysis['lexical_score'],
        'anchor_lexical_score': anchor_analysis['lexical_score'],
        'base_negative_total': base_analysis['negative_total'],
        'anchor_negative_total': anchor_analysis['negative_total'],
        'active_bias_steps': active_steps,
        'mean_alpha_t': mean_alpha,
        'mean_entropy': mean_entropy,
        'mean_pressure': mean_pressure,
    }
    all_results.append({'summary': summary, 'payload': payload})

    print('base lexical score:', base_analysis['lexical_score'])
    print('anchor lexical score:', anchor_analysis['lexical_score'])
    print('base negative hits:', base_analysis['negative_total'])
    print('anchor negative hits:', anchor_analysis['negative_total'])
    print('anchor bias active steps:', active_steps)
    print('mean alpha_t:', mean_alpha)
    print('mean entropy:', mean_entropy)
    print('mean pressure:', mean_pressure)
    print('saved json:', json_path)
    print('saved md:', md_path)


In [ ]:
for item in all_results:
    s = item['summary']
    print(f"{s['name']}: base={s['base_lexical_score']:.2f} | anchor={s['anchor_lexical_score']:.2f} | "+
          f"base_neg={s['base_negative_total']} | anchor_neg={s['anchor_negative_total']} | "+
          f"active_steps={s['active_bias_steps']} | mean_alpha={s['mean_alpha_t']:.4f} | mean_pressure={s['mean_pressure']:.4f}")


In [ ]:
for item in all_results:
    payload = item['payload']
    print(f"\n########## {payload['run_name']} ##########")
    print('prompt:', payload['prompt'])
    print('--- base continuation ---')
    print(payload['base']['continuation_text'][:3000])
    print('--- anchor continuation ---')
    print(payload['anchor']['continuation_text'][:3000])
    print('--- first 25 anchor steps ---')
    for idx, step in enumerate(payload['anchor']['steps'][:25], start=1):
        print(
            idx,
            repr(step.get('token_text')),
            'alpha=', round(step.get('effective_bias_scale', 0.0), 4),
            'entropy=', round(step.get('normalized_entropy', 0.0), 4),
            'pressure=', round(step.get('raw_contradiction_pressure', 0.0), 4),
            'bias=', step.get('bias_applied'),
        )
        if step.get('top_biased_tokens'):
            print('  top biased:', step['top_biased_tokens'][:3])


In [ ]:
for item in all_results:
    md_path = item['summary']['md_path']
    print(f"\n########## report: {item['summary']['name']} ##########")
    print(Path(md_path).read_text(encoding='utf-8')[:4000])
